# Setup Bronze tables

In [0]:
%reload_ext autoreload
%autoreload 2

import importlib
import etl_config.bronze_config as bronze_config_module

importlib.reload(bronze_config_module)

# Bronze table configuration

- Should be executed after first job run
- Compact existing small files
- Set table properties to enable auto compaction and optimize future writes
- Set columns description

In [0]:
from utils.logger import get_logger
from etl_config.bronze_config import (
    CATALOG,
    BRONZE,
    BRONZE_CONFIG,
)

logger = get_logger("bronze_setup")

In [0]:
for table_name, cfg in BRONZE_CONFIG.items():
    # Compact existing small files
    spark.sql(f"OPTIMIZE {cfg.table_name}")
    logger.info(f"Optimized table {cfg.table_name}")

    # Set table properties to enable auto compaction and optimize future writes
    spark.sql(f"""
        ALTER TABLE {cfg.table_name}
        SET TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true',
            'description' = '{cfg.table_description}'
        )
    """)
    logger.info(f"Set table properties for {cfg.table_name} finished.")

    for column, description in cfg.column_comments.items():
        spark.sql(f"""
            COMMENT ON COLUMN {cfg.table_name}.{column}
            IS '{description}'
        """)

    logger.info(f"Commented columns for {cfg.table_name} finished.")